# AI-Powered Industrial Defect Detection — Part 2: Flask Web App

Runs the trained model from Part 1 as a web app with:
- **Upload inspection** (single or batch) with confidence + Grad-CAM heatmap
- **Live webcam inspection** (real-time, simulates a conveyor-belt camera)
- **Dashboard** with defect rate, charts and history (SQLite), plus CSV export



## 1. Connect Google Drive and check the trained model

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/defect-detection'
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
for f in ['defect_model.keras', 'model_config.json']:
    path = os.path.join(MODEL_DIR, f)
    print(('✅ ' if os.path.exists(path) else '❌ MISSING: ') + path)

Mounted at /content/drive
✅ /content/drive/MyDrive/defect-detection/models/defect_model.keras
✅ /content/drive/MyDrive/defect-detection/models/model_config.json


In [2]:
!pip install -q flask
!mkdir -p /content/defect_app/templates /content/defect_app/static

## 2. Write the app files
Each cell below creates one file of the Flask app inside `/content/defect_app/`.

In [3]:
%%writefile /content/defect_app/model_utils.py
"""Model loading, OpenCV preprocessing, prediction and Grad-CAM for the defect detection app."""
import json
import os
import threading
import time

import cv2
import keras
import numpy as np
import tensorflow as tf


class DefectDetector:
    def __init__(self, model_dir):
        with open(os.path.join(model_dir, 'model_config.json')) as f:
            self.config = json.load(f)
        self.model = keras.models.load_model(os.path.join(model_dir, 'defect_model.keras'))
        self.img_size = tuple(self.config['img_size'])
        self.threshold = float(self.config['threshold'])
        self.class_names = self.config['class_names']          # ['OK', 'DEFECTIVE']

        # Layers used to run the model in pieces, so one pass gives both the prediction and Grad-CAM
        self._rescale = self.model.get_layer('rescale')
        self._base = self.model.get_layer(self.config['base_model_layer'])
        self._gap = self.model.get_layer('gap')
        self._pred = self.model.get_layer('pred')
        self._lock = threading.Lock()

        self.predict(np.zeros(self.img_size + (3,), np.uint8))  # warm-up so the first real request is fast

    def preprocess(self, image_bgr):
        """BGR image (cv2.imread / webcam frame) -> float32 batch (1, H, W, 3), raw RGB 0-255."""
        img = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size[1], self.img_size[0]), interpolation=cv2.INTER_LINEAR)
        return np.expand_dims(img.astype('float32'), axis=0)

    def predict(self, image_bgr):
        x = self.preprocess(image_bgr)
        start = time.perf_counter()
        with self._lock:
            conv_out = self._base(self._rescale(x), training=False)
            with tf.GradientTape() as tape:
                tape.watch(conv_out)
                pred = self._pred(self._gap(conv_out))
                score = pred[:, 0]
            grads = tape.gradient(score, conv_out)
        weights = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.reduce_sum(conv_out[0] * weights, axis=-1)
        heatmap = (tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)).numpy()
        elapsed_ms = (time.perf_counter() - start) * 1000

        defect_prob = float(pred[0, 0])
        is_defective = defect_prob >= self.threshold
        return {
            'label': self.class_names[int(is_defective)],
            'is_defective': bool(is_defective),
            'defect_prob': defect_prob,
            'confidence': defect_prob if is_defective else 1 - defect_prob,
            'inference_ms': elapsed_ms,
            'heatmap': heatmap,
        }


def overlay_heatmap(image_bgr, heatmap, alpha=0.4):
    h, w = image_bgr.shape[:2]
    hm = cv2.resize(heatmap, (w, h))
    hm = cv2.applyColorMap(np.uint8(255 * hm), cv2.COLORMAP_JET)
    return cv2.addWeighted(image_bgr, 1 - alpha, hm, alpha, 0)


def decode_image(data):
    """Raw file bytes -> BGR image, or None if the bytes are not a valid image."""
    arr = np.frombuffer(data, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR) if arr.size else None

Writing /content/defect_app/model_utils.py


In [4]:
%%writefile /content/defect_app/database.py
"""SQLite storage for the inspection history."""
import sqlite3
from datetime import datetime


class InspectionDB:
    def __init__(self, path):
        self.path = path
        with self._connect() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS inspections (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    timestamp TEXT NOT NULL,
                    source TEXT NOT NULL,          -- 'upload' or 'live'
                    filename TEXT,
                    label TEXT NOT NULL,           -- 'OK' or 'DEFECTIVE'
                    defect_prob REAL NOT NULL,
                    confidence REAL NOT NULL,
                    inference_ms REAL NOT NULL,
                    image_file TEXT,
                    heatmap_file TEXT
                )""")

    def _connect(self):
        conn = sqlite3.connect(self.path)
        conn.row_factory = sqlite3.Row
        return conn

    def log(self, source, filename, result, image_file=None, heatmap_file=None):
        with self._connect() as conn:
            cur = conn.execute(
                """INSERT INTO inspections (timestamp, source, filename, label, defect_prob, confidence,
                                            inference_ms, image_file, heatmap_file)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (datetime.now().strftime('%Y-%m-%d %H:%M:%S'), source, filename, result['label'],
                 result['defect_prob'], result['confidence'], result['inference_ms'], image_file, heatmap_file))
            return cur.lastrowid

    def stats(self):
        with self._connect() as conn:
            row = conn.execute("""
                SELECT COUNT(*) AS total,
                       COALESCE(SUM(label = 'DEFECTIVE'), 0) AS defective,
                       COALESCE(AVG(confidence), 0) AS avg_confidence,
                       COALESCE(AVG(inference_ms), 0) AS avg_ms
                FROM inspections""").fetchone()
            daily = conn.execute("""
                SELECT substr(timestamp, 1, 10) AS day,
                       SUM(label = 'OK') AS ok, SUM(label = 'DEFECTIVE') AS defective
                FROM inspections GROUP BY day ORDER BY day DESC LIMIT 14""").fetchall()
        stats = dict(row)
        stats['ok'] = stats['total'] - stats['defective']
        stats['defect_rate'] = 100 * stats['defective'] / stats['total'] if stats['total'] else 0
        stats['daily'] = [dict(r) for r in reversed(daily)]
        return stats

    def recent(self, limit=50):
        with self._connect() as conn:
            return [dict(r) for r in conn.execute(
                'SELECT * FROM inspections ORDER BY id DESC LIMIT ?', (limit,)).fetchall()]

    def all_rows(self):
        with self._connect() as conn:
            return [dict(r) for r in conn.execute('SELECT * FROM inspections ORDER BY id').fetchall()]

Writing /content/defect_app/database.py


In [5]:
%%writefile /content/defect_app/app.py
"""Flask web app for AI-powered industrial defect detection.

Environment variables:
  MODEL_DIR  folder with defect_model.keras + model_config.json   (default: ./models)
  DATA_DIR   folder for the SQLite database and saved images       (default: ./data)
  PORT       port to listen on                                     (default: 5000)
"""
import base64
import csv
import io
import os
import uuid

import cv2
from flask import Flask, Response, jsonify, redirect, render_template, request, send_from_directory, url_for

from database import InspectionDB
from model_utils import DefectDetector, decode_image, overlay_heatmap

APP_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.environ.get('MODEL_DIR', os.path.join(APP_DIR, 'models'))
DATA_DIR = os.environ.get('DATA_DIR', os.path.join(APP_DIR, 'data'))
MEDIA_DIR = os.path.join(DATA_DIR, 'media')
os.makedirs(MEDIA_DIR, exist_ok=True)

ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 32 * 1024 * 1024   # 32 MB per request

print('Loading model from', MODEL_DIR)
detector = DefectDetector(MODEL_DIR)
db = InspectionDB(os.path.join(DATA_DIR, 'inspections.db'))
print('Model ready. Decision threshold =', detector.threshold)


def save_result_images(image_bgr, result):
    """Save the original and the Grad-CAM overlay; return their file names inside MEDIA_DIR."""
    key = uuid.uuid4().hex
    image_file, heatmap_file = f'{key}.jpg', f'{key}_cam.jpg'
    cv2.imwrite(os.path.join(MEDIA_DIR, image_file), image_bgr)
    cv2.imwrite(os.path.join(MEDIA_DIR, heatmap_file), overlay_heatmap(image_bgr, result['heatmap']))
    return image_file, heatmap_file


def to_data_url(image_bgr):
    ok, buf = cv2.imencode('.jpg', image_bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return 'data:image/jpeg;base64,' + base64.b64encode(buf).decode()


@app.route('/')
def index():
    return render_template('index.html', threshold=detector.threshold)


@app.route('/inspect', methods=['POST'])
def inspect():
    results, errors = [], []
    for file in request.files.getlist('images'):
        if not file or not file.filename:
            continue
        if os.path.splitext(file.filename)[1].lower() not in ALLOWED_EXTENSIONS:
            errors.append(f'{file.filename}: unsupported file type')
            continue
        image = decode_image(file.read())
        if image is None:
            errors.append(f'{file.filename}: could not read image')
            continue
        result = detector.predict(image)
        image_file, heatmap_file = save_result_images(image, result)
        db.log('upload', file.filename, result, image_file, heatmap_file)
        result.update(filename=file.filename, image_file=image_file, heatmap_file=heatmap_file)
        results.append(result)
    if not results and not errors:
        return redirect(url_for('index'))
    summary = {'total': len(results), 'defective': sum(r['is_defective'] for r in results)}
    return render_template('result.html', results=results, errors=errors, summary=summary)


@app.route('/live')
def live():
    return render_template('live.html', threshold=detector.threshold)


@app.route('/api/predict', methods=['POST'])
def api_predict():
    """JSON in: {"image": "<data URL or base64>", "log": bool}. Used by the live webcam page."""
    payload = request.get_json(silent=True) or {}
    data = payload.get('image', '')
    if ',' in data:
        data = data.split(',', 1)[1]
    try:
        image = decode_image(base64.b64decode(data))
    except (ValueError, TypeError):
        image = None
    if image is None:
        return jsonify(error='invalid image'), 400

    result = detector.predict(image)
    if payload.get('log'):
        image_file, heatmap_file = save_result_images(image, result)
        db.log('live', 'webcam frame', result, image_file, heatmap_file)
    return jsonify(
        label=result['label'],
        is_defective=result['is_defective'],
        defect_prob=round(result['defect_prob'], 4),
        confidence=round(result['confidence'], 4),
        inference_ms=round(result['inference_ms'], 1),
        heatmap=to_data_url(overlay_heatmap(image, result['heatmap'])),
    )


@app.route('/dashboard')
def dashboard():
    return render_template('dashboard.html', stats=db.stats(), rows=db.recent(50))


@app.route('/export.csv')
def export_csv():
    out = io.StringIO()
    writer = csv.writer(out)
    writer.writerow(['id', 'timestamp', 'source', 'filename', 'label', 'defect_prob', 'confidence', 'inference_ms'])
    for r in db.all_rows():
        writer.writerow([r['id'], r['timestamp'], r['source'], r['filename'], r['label'],
                         f"{r['defect_prob']:.4f}", f"{r['confidence']:.4f}", f"{r['inference_ms']:.1f}"])
    return Response(out.getvalue(), mimetype='text/csv',
                    headers={'Content-Disposition': 'attachment; filename=inspection_history.csv'})


@app.route('/media/<path:filename>')
def media(filename):
    return send_from_directory(MEDIA_DIR, filename)


@app.route('/health')
def health():
    return jsonify(status='ok', threshold=detector.threshold)


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=int(os.environ.get('PORT', 5000)), debug=False, threaded=True)

Writing /content/defect_app/app.py


In [6]:
%%writefile /content/defect_app/templates/base.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{% block title %}Defect Inspector{% endblock %}</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
  <link href="{{ url_for('static', filename='style.css') }}" rel="stylesheet">
</head>
<body>
<nav class="navbar navbar-expand-md navbar-dark bg-dark mb-4">
  <div class="container">
    <a class="navbar-brand fw-semibold" href="{{ url_for('index') }}">🔍 Defect Inspector</a>
    <button class="navbar-toggler" type="button" data-bs-toggle="collapse" data-bs-target="#nav">
      <span class="navbar-toggler-icon"></span>
    </button>
    <div class="collapse navbar-collapse" id="nav">
      <ul class="navbar-nav ms-auto">
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'index' %}active{% endif %}" href="{{ url_for('index') }}">Upload</a></li>
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'live' %}active{% endif %}" href="{{ url_for('live') }}">Live Inspection</a></li>
        <li class="nav-item"><a class="nav-link {% if request.endpoint == 'dashboard' %}active{% endif %}" href="{{ url_for('dashboard') }}">Dashboard</a></li>
      </ul>
    </div>
  </div>
</nav>
<main class="container pb-5">
  {% block content %}{% endblock %}
</main>
<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/js/bootstrap.bundle.min.js"></script>
{% block scripts %}{% endblock %}
</body>
</html>

Writing /content/defect_app/templates/base.html


In [7]:
%%writefile /content/defect_app/templates/index.html
{% extends 'base.html' %}
{% block content %}
<div class="row justify-content-center">
  <div class="col-lg-8">
    <h1 class="h3 mb-2">Visual Quality Inspection</h1>
    <p class="text-secondary">Upload one or more images of casting parts. The AI model (MobileNetV2) classifies each part as
      <span class="badge bg-success">OK</span> or <span class="badge bg-danger">DEFECTIVE</span> and shows a Grad-CAM heatmap of where it found the defect.</p>

    <form action="{{ url_for('inspect') }}" method="post" enctype="multipart/form-data" class="card card-body shadow-sm" id="upload-form">
      <label for="images" class="drop-zone" id="drop-zone">
        <div class="fs-1">📤</div>
        <div class="fw-semibold">Click to choose images or drag them here</div>
        <div class="small text-secondary">JPG, PNG or BMP · multiple files allowed</div>
        <div class="small mt-2" id="file-count"></div>
      </label>
      <input type="file" name="images" id="images" accept=".jpg,.jpeg,.png,.bmp" multiple required hidden>
      <button type="submit" class="btn btn-primary btn-lg mt-3" id="submit-btn">Inspect</button>
    </form>

    <p class="small text-secondary mt-3">Decision threshold: P(defect) ≥ {{ '%.2f'|format(threshold) }} → DEFECTIVE
      (tuned in training to keep defect recall high).</p>
  </div>
</div>
{% endblock %}
{% block scripts %}
<script>
  const input = document.getElementById('images');
  const zone = document.getElementById('drop-zone');
  const count = document.getElementById('file-count');
  const showCount = () => { count.textContent = input.files.length ? `${input.files.length} file(s) selected` : ''; };
  input.addEventListener('change', showCount);
  ['dragover', 'dragenter'].forEach(e => zone.addEventListener(e, ev => { ev.preventDefault(); zone.classList.add('hover'); }));
  ['dragleave', 'drop'].forEach(e => zone.addEventListener(e, () => zone.classList.remove('hover')));
  zone.addEventListener('drop', ev => { ev.preventDefault(); input.files = ev.dataTransfer.files; showCount(); });
  document.getElementById('upload-form').addEventListener('submit', () => {
    const btn = document.getElementById('submit-btn');
    btn.disabled = true; btn.textContent = 'Inspecting…';
  });
</script>
{% endblock %}

Writing /content/defect_app/templates/index.html


In [8]:
%%writefile /content/defect_app/templates/result.html
{% extends 'base.html' %}
{% block title %}Results · Defect Inspector{% endblock %}
{% block content %}
<div class="d-flex flex-wrap justify-content-between align-items-center mb-3 gap-2">
  <h1 class="h3 mb-0">Inspection results</h1>
  <a href="{{ url_for('index') }}" class="btn btn-outline-primary">Inspect more</a>
</div>

{% if summary.total > 1 %}
<div class="alert alert-secondary">
  {{ summary.total }} parts inspected ·
  <strong class="text-danger">{{ summary.defective }} defective</strong> ·
  <strong class="text-success">{{ summary.total - summary.defective }} OK</strong>
</div>
{% endif %}

{% for e in errors %}
<div class="alert alert-warning py-2">{{ e }}</div>
{% endfor %}

<div class="row g-4">
  {% for r in results %}
  <div class="col-md-6 col-xl-4">
    <div class="card shadow-sm h-100 border-{{ 'danger' if r.is_defective else 'success' }}">
      <div class="card-header d-flex justify-content-between align-items-center">
        <span class="text-truncate small" title="{{ r.filename }}">{{ r.filename }}</span>
        <span class="badge fs-6 bg-{{ 'danger' if r.is_defective else 'success' }}">{{ r.label }}</span>
      </div>
      <div class="row g-0">
        <div class="col-6 p-1 text-center">
          <img src="{{ url_for('media', filename=r.image_file) }}" class="img-fluid rounded" alt="original">
          <div class="small text-secondary">Original</div>
        </div>
        <div class="col-6 p-1 text-center">
          <img src="{{ url_for('media', filename=r.heatmap_file) }}" class="img-fluid rounded" alt="Grad-CAM">
          <div class="small text-secondary">Grad-CAM</div>
        </div>
      </div>
      <div class="card-body">
        <div class="d-flex justify-content-between small mb-1">
          <span>Confidence</span><strong>{{ '%.1f'|format(r.confidence * 100) }}%</strong>
        </div>
        <div class="progress mb-2" style="height: 8px">
          <div class="progress-bar bg-{{ 'danger' if r.is_defective else 'success' }}" style="width: {{ r.confidence * 100 }}%"></div>
        </div>
        <div class="small text-secondary">P(defect) = {{ '%.4f'|format(r.defect_prob) }} · {{ '%.0f'|format(r.inference_ms) }} ms</div>
      </div>
    </div>
  </div>
  {% endfor %}
</div>
{% endblock %}

Writing /content/defect_app/templates/result.html


In [9]:
%%writefile /content/defect_app/templates/live.html
{% extends 'base.html' %}
{% block title %}Live Inspection · Defect Inspector{% endblock %}
{% block content %}
<h1 class="h3 mb-2">Live inspection</h1>
<p class="text-secondary">Simulates a camera above a conveyor belt: frames from your webcam are sent to the model a few times per second.
  Hold a part (or a photo of one on your phone/screen) in front of the camera.</p>

<div class="row g-4">
  <div class="col-lg-6">
    <div class="card shadow-sm">
      <div class="card-header">Camera</div>
      <div class="card-body text-center">
        <video id="video" autoplay playsinline muted class="w-100 rounded bg-dark"></video>
        <div class="d-flex flex-wrap gap-2 justify-content-center mt-3">
          <button id="start" class="btn btn-success">Start inspection</button>
          <button id="stop" class="btn btn-outline-danger" disabled>Stop</button>
          <div class="form-check form-switch align-self-center ms-2">
            <input class="form-check-input" type="checkbox" id="log">
            <label class="form-check-label" for="log">Save results to history</label>
          </div>
        </div>
        <div id="cam-error" class="alert alert-warning mt-3 d-none"></div>
      </div>
    </div>
  </div>
  <div class="col-lg-6">
    <div class="card shadow-sm">
      <div class="card-header d-flex justify-content-between align-items-center">
        <span>Model output</span>
        <span id="label" class="badge bg-secondary fs-5">—</span>
      </div>
      <div class="card-body text-center">
        <img id="heatmap" class="w-100 rounded bg-light" alt="Grad-CAM" style="min-height: 200px">
        <div class="row mt-3 text-center">
          <div class="col"><div class="small text-secondary">Confidence</div><div id="conf" class="fs-5 fw-semibold">—</div></div>
          <div class="col"><div class="small text-secondary">P(defect)</div><div id="prob" class="fs-5 fw-semibold">—</div></div>
          <div class="col"><div class="small text-secondary">Inference</div><div id="ms" class="fs-5 fw-semibold">—</div></div>
          <div class="col"><div class="small text-secondary">Frames</div><div id="frames" class="fs-5 fw-semibold">0</div></div>
        </div>
      </div>
    </div>
  </div>
</div>
<canvas id="canvas" width="320" height="320" hidden></canvas>
{% endblock %}
{% block scripts %}
<script>
  const video = document.getElementById('video'), canvas = document.getElementById('canvas');
  const startBtn = document.getElementById('start'), stopBtn = document.getElementById('stop');
  const els = id => document.getElementById(id);
  let stream = null, running = false, frames = 0;

  async function start() {
    try {
      stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
    } catch (err) {
      const box = els('cam-error');
      box.textContent = 'Could not open the camera: ' + err.message + '. Allow camera access in your browser and try again.';
      box.classList.remove('d-none');
      return;
    }
    video.srcObject = stream;
    running = true; startBtn.disabled = true; stopBtn.disabled = false;
    loop();
  }

  function stop() {
    running = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    startBtn.disabled = false; stopBtn.disabled = true;
  }

  async function loop() {
    if (!running) return;
    if (video.videoWidth) {
      // Centre-crop a square from the video frame, like the square training images
      const s = Math.min(video.videoWidth, video.videoHeight);
      const ctx = canvas.getContext('2d');
      ctx.drawImage(video, (video.videoWidth - s) / 2, (video.videoHeight - s) / 2, s, s, 0, 0, canvas.width, canvas.height);
      try {
        const res = await fetch('{{ url_for("api_predict") }}', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ image: canvas.toDataURL('image/jpeg', 0.85), log: els('log').checked }),
        });
        const r = await res.json();
        if (res.ok) {
          const label = els('label');
          label.textContent = r.label;
          label.className = 'badge fs-5 bg-' + (r.is_defective ? 'danger' : 'success');
          els('heatmap').src = r.heatmap;
          els('conf').textContent = (r.confidence * 100).toFixed(1) + '%';
          els('prob').textContent = r.defect_prob.toFixed(3);
          els('ms').textContent = r.inference_ms.toFixed(0) + ' ms';
          els('frames').textContent = ++frames;
        }
      } catch (err) {
        console.error(err);
      }
    }
    setTimeout(loop, 300);
  }

  startBtn.addEventListener('click', start);
  stopBtn.addEventListener('click', stop);
</script>
{% endblock %}

Writing /content/defect_app/templates/live.html


In [10]:
%%writefile /content/defect_app/templates/dashboard.html
{% extends 'base.html' %}
{% block title %}Dashboard · Defect Inspector{% endblock %}
{% block content %}
<div class="d-flex flex-wrap justify-content-between align-items-center mb-3 gap-2">
  <h1 class="h3 mb-0">Inspection dashboard</h1>
  <a href="{{ url_for('export_csv') }}" class="btn btn-outline-secondary btn-sm">⬇ Export CSV</a>
</div>

<div class="row g-3 mb-4">
  <div class="col-6 col-lg-3"><div class="card stat-card"><div class="card-body">
    <div class="small text-secondary">Parts inspected</div><div class="fs-3 fw-bold">{{ stats.total }}</div></div></div></div>
  <div class="col-6 col-lg-3"><div class="card stat-card"><div class="card-body">
    <div class="small text-secondary">Defect rate</div><div class="fs-3 fw-bold text-danger">{{ '%.1f'|format(stats.defect_rate) }}%</div></div></div></div>
  <div class="col-6 col-lg-3"><div class="card stat-card"><div class="card-body">
    <div class="small text-secondary">Avg confidence</div><div class="fs-3 fw-bold">{{ '%.1f'|format(stats.avg_confidence * 100) }}%</div></div></div></div>
  <div class="col-6 col-lg-3"><div class="card stat-card"><div class="card-body">
    <div class="small text-secondary">Avg inference time</div><div class="fs-3 fw-bold">{{ '%.0f'|format(stats.avg_ms) }} ms</div></div></div></div>
</div>

{% if stats.total %}
<div class="row g-4 mb-4">
  <div class="col-lg-4"><div class="card shadow-sm h-100"><div class="card-header">OK vs defective</div>
    <div class="card-body"><canvas id="pie"></canvas></div></div></div>
  <div class="col-lg-8"><div class="card shadow-sm h-100"><div class="card-header">Inspections per day</div>
    <div class="card-body"><canvas id="daily"></canvas></div></div></div>
</div>

<div class="card shadow-sm">
  <div class="card-header">Recent inspections (last 50)</div>
  <div class="table-responsive">
    <table class="table table-sm table-hover align-middle mb-0">
      <thead><tr><th>#</th><th>Time</th><th>Source</th><th>File</th><th>Result</th><th>Confidence</th><th>ms</th><th>Image</th></tr></thead>
      <tbody>
      {% for r in rows %}
        <tr>
          <td>{{ r.id }}</td>
          <td class="text-nowrap small">{{ r.timestamp }}</td>
          <td><span class="badge bg-light text-dark border">{{ r.source }}</span></td>
          <td class="small text-truncate" style="max-width: 180px">{{ r.filename }}</td>
          <td><span class="badge bg-{{ 'danger' if r.label == 'DEFECTIVE' else 'success' }}">{{ r.label }}</span></td>
          <td>{{ '%.1f'|format(r.confidence * 100) }}%</td>
          <td>{{ '%.0f'|format(r.inference_ms) }}</td>
          <td>{% if r.heatmap_file %}<a href="{{ url_for('media', filename=r.heatmap_file) }}" target="_blank">
            <img src="{{ url_for('media', filename=r.image_file) }}" class="thumb rounded" alt=""></a>{% endif %}</td>
        </tr>
      {% endfor %}
      </tbody>
    </table>
  </div>
</div>
{% else %}
<div class="alert alert-info">No inspections yet. <a href="{{ url_for('index') }}">Upload some images</a> or try <a href="{{ url_for('live') }}">live inspection</a>.</div>
{% endif %}
{% endblock %}
{% block scripts %}
{% if stats.total %}
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<script>
  const stats = {{ stats|tojson }};
  new Chart(document.getElementById('pie'), {
    type: 'doughnut',
    data: { labels: ['OK', 'DEFECTIVE'],
            datasets: [{ data: [stats.ok, stats.defective], backgroundColor: ['#198754', '#dc3545'] }] },
  });
  new Chart(document.getElementById('daily'), {
    type: 'bar',
    data: { labels: stats.daily.map(d => d.day),
            datasets: [{ label: 'OK', data: stats.daily.map(d => d.ok), backgroundColor: '#198754' },
                       { label: 'DEFECTIVE', data: stats.daily.map(d => d.defective), backgroundColor: '#dc3545' }] },
    options: { scales: { x: { stacked: true }, y: { stacked: true, beginAtZero: true, ticks: { precision: 0 } } } },
  });
</script>
{% endif %}
{% endblock %}

Writing /content/defect_app/templates/dashboard.html


In [11]:
%%writefile /content/defect_app/static/style.css
body { background: #f5f7fa; }

.drop-zone {
  display: block;
  border: 2px dashed #adb5bd;
  border-radius: 12px;
  padding: 2.5rem 1rem;
  text-align: center;
  cursor: pointer;
  background: #fff;
  transition: border-color .15s, background .15s;
}
.drop-zone:hover, .drop-zone.hover { border-color: #0d6efd; background: #f0f6ff; }

.stat-card { border: none; box-shadow: 0 1px 3px rgba(0, 0, 0, .08); }

.thumb { width: 44px; height: 44px; object-fit: cover; }

#video, #heatmap { max-height: 380px; object-fit: contain; }

Writing /content/defect_app/static/style.css


In [12]:
%%writefile /content/defect_app/requirements.txt
flask>=3.0
tensorflow>=2.16
opencv-python-headless>=4.8
numpy

Writing /content/defect_app/requirements.txt


In [13]:
%%writefile /content/defect_app/README.md
# Defect Inspector — Flask web app

Web interface for the AI-powered industrial defect detection model (MobileNetV2, trained in Part 1).

## Features
- **Upload inspection** — upload one or many images and get OK / DEFECTIVE, confidence and a Grad-CAM heatmap for each
- **Live inspection** — classifies webcam frames in real time (simulates a camera on a conveyor belt)
- **Dashboard** — total inspected, defect rate, average confidence and inference time, charts, recent history
- **History** — every inspection stored in SQLite, exportable to CSV

## Project structure
```
defect_app/
├── app.py            # Flask routes
├── model_utils.py    # model loading, OpenCV preprocessing, prediction, Grad-CAM
├── database.py       # SQLite inspection history
├── templates/        # base, index (upload), result, live, dashboard
├── static/style.css
├── models/           # put defect_model.keras + model_config.json here (when running locally)
└── requirements.txt
```

## Run on your laptop
1. Copy `defect_model.keras` and `model_config.json` from `MyDrive/defect-detection/models/` into `defect_app/models/`.
2. Install the requirements (Python 3.10–3.12):
   ```bash
   pip install -r requirements.txt
   ```
3. Start the app:
   ```bash
   python app.py
   ```
4. Open http://localhost:5000

Use the same TensorFlow version as Colab (check with `tf.__version__` there) if loading the model fails.

## API
`POST /api/predict` with JSON `{"image": "<base64 or data URL>", "log": false}` returns
`{label, is_defective, defect_prob, confidence, inference_ms, heatmap}`.

Writing /content/defect_app/README.md


## 3. Start the server
The inspection history is kept in `/content/app_data` while the app runs (SQLite works best on local disk).
If you backed it up to Drive in an earlier session (step 6), it is restored first.

In [22]:
import shutil, subprocess, sys, time, requests

APP_DIR = '/content/defect_app'
DATA_DIR = '/content/app_data'
DRIVE_DATA = os.path.join(PROJECT_DIR, 'app_data')
if not os.path.exists(DATA_DIR) and os.path.exists(DRIVE_DATA):
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print('Restored inspection history from Drive')

subprocess.run(['pkill', '-f', 'app.py'])                 # stop an old server if this cell is re-run
env = dict(os.environ, MODEL_DIR=MODEL_DIR, DATA_DIR=DATA_DIR, PORT='5000')
log_file = open('/content/server.log', 'w')
server = subprocess.Popen([sys.executable, 'app.py'], cwd=APP_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT)

print('Starting server (loading the model takes ~10-30 s)...')
for _ in range(120):
    try:
        print('Server is up:', requests.get('http://127.0.0.1:5000/health', timeout=2).json())
        break
    except requests.exceptions.ConnectionError:
        if server.poll() is not None:
            print('❌ Server crashed. Log:'); print(open('/content/server.log').read()); break
        time.sleep(1)

Starting server (loading the model takes ~10-30 s)...
Server is up: {'status': 'ok', 'threshold': 0.8}


## 4. Open the app
Click the link printed below — the app opens in a new browser tab.
On the **Live Inspection** page, allow camera access when the browser asks.

In [23]:
from google.colab import output
output.serve_kernel_port_as_window(5000, anchor_text='👉 Open Defect Inspector')

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [18]:
# Server log (useful if something goes wrong)
!tail -n 30 /content/server.log

I0000 00:00:1793328543.146056    1539 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1793328555.933896    1539 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1793328555.935789    1539 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793328559.085165    1539 cuda_dnn.cc:461] Loaded cuDNN version 92700
Loading model from /content/drive/MyDrive/defect-detection/models
Model ready. Decision threshold = 0.8
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on